# 수출 유망국 대시보드 가설검정

`02_EDA.ipynb`에서 확인한 지표 관계를 바탕으로, 대시보드 차트 설계에 필요한 판단을 검정으로 확인

| # | 검정 | 답하려는 질문 |
|---|---|---|
| 1 | 준비 | 데이터 로드, 공통 함수 |
| 2 | 축 조합 검정 | 버블차트의 x축과 y축은 관계가 있는가, 두 차트는 서로 다른 정보를 보여주는가 |
| 3 | 버블 크기 검정 | 버블 크기로 쓸 지표가 x축과 겹치지 않는가 |
| 4 | 추세 검정 | 어느 나라의 지표가 일관되게 늘거나 줄고 있는가 |

**공통 방침 (결과를 보기 전에 고정)**

- 유의수준 `ALPHA = 0.05`
- 등가한계 `EQUIV_BOUND = 0.3`
- 상관은 Spearman 순위상관 사용 (원본 지표의 정규성이 모두 기각됐기 때문)
- 같은 질문에 대한 검정을 여러 개 한꺼번에 할 때는 그 묶음 안에서 Benjamini-Hochberg FDR 보정을 하고, 보정된 값(q)으로 판정

In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

sns.set_theme(style='whitegrid')

# 한글 폰트 (02_EDA랑 동일)
installed = {f.name for f in font_manager.fontManager.ttflist}
font = next((f for f in ['Noto Sans KR', 'Malgun Gothic', 'NanumGothic', 'AppleGothic']
             if f in installed), None)
if font:
    plt.rcParams['font.family'] = font
else:
    print('한글 폰트 없음. 그래프 한글이 깨질 수 있음')
plt.rcParams['axes.unicode_minus'] = False

---
## 1. 준비

In [2]:
# 기준값은 분석 전에 미리 고정
# 결과를 보고 나서 기준을 바꾸면 원하는 결론에 맞추게 되니까 검정 전에 정해둠
ALPHA = 0.05         # 유의수준
EQUIV_BOUND = 0.3   # 등가 한계. |rho|가 이보다 작으면 실무적으로 무관하다고 봄 (TOST)
YEAR = 2023          # 기준 연도
MIN_OBS = 5          # 추세 검정 최소 관측 수 (이유는 4절 mann_kendall 주석에)

# 지표, 한글 이름 (02_EDA랑 동일)
cols = ['current_usd', 'gdp_calculated', 'TIV_5Y_Share', 'human_hazard_score', 'share_gdp']

LABEL = {
    'current_usd': '군사비',             # 경상 달러
    'gdp_calculated': 'GDP',
    'TIV_5Y_Share': '수입점유율',        # 최근 5년 TIV 세계 점유율(%)
    'human_hazard_score': '분쟁위험도',  # 0~10, 높을수록 위험
    'share_gdp': '군사비/GDP',           # 0~1 비율
}

In [3]:
# DB 접속 (02_EDA랑 동일). 접속 정보는 .env에서 읽음
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()), Path.cwd())
env_file = ROOT / '.env'
if env_file.exists():
    for line in env_file.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

missing = [k for k in ['DB_USER', 'DB_PASSWORD', 'DB_HOST', 'DB_NAME'] if not os.environ.get(k)]
if missing:
    raise RuntimeError(f'환경변수 없음: {", ".join(missing)} (.env 파일 확인)')

ssl_ca = Path(os.environ.get('DB_SSL_CA', './global-bundle.pem'))
if not ssl_ca.is_absolute():
    ssl_ca = ROOT / ssl_ca   # 상대경로면 루트 기준
if not ssl_ca.exists():
    raise FileNotFoundError(f'RDS 인증서 없음: {ssl_ca}')

db_connection_url = URL.create(
    'mysql+pymysql',
    username=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
    host=os.environ['DB_HOST'],
    port=int(os.environ.get('DB_PORT', 3306)),
    database=os.environ['DB_NAME'],
)
engine = create_engine(db_connection_url, connect_args={'ssl': {'ca': str(ssl_ca)}})

df_demand = pd.read_sql("SELECT * FROM integrate", con=engine)
print(df_demand.shape)

(4772, 9)


In [4]:
# 기준 연도에 지표 5개가 다 있는 나라만
d = df_demand[df_demand.Year == YEAR].dropna(subset=cols).copy()
N = len(d)

print(f'{YEAR}년 분석 대상: {N}개국')
d[cols].describe().round(2)

2023년 분석 대상: 147개국


,current_usd,gdp_calculated,TIV_5Y_Share,human_hazard_score,share_gdp
count,147.00,147.00,147.00,147.00,147.00
mean,16017.79,702268.65,0.60,3.71,0.02
std,80426.76,2808534.65,1.40,3.15,0.03
min,11.79,2080.00,0.00,0.00,0.00
25%,285.62,19948.46,0.02,0.90,0.01
50%,1207.28,76370.39,0.08,3.10,0.02
75%,5734.85,373866.27,0.47,6.60,0.02
max,916014.70,27811517.00,10.23,9.80,0.36


In [5]:
# 버블 크기 후보 (log10). 금액은 나라별 자릿수 차이가 커서 로그로 (0 이하는 NaN)
# 검정은 전부 Spearman이라 로그를 취해도 결과는 같음
d['log_mil'] = np.log10(d.current_usd.where(d.current_usd > 0))
d['log_gdp'] = np.log10(d.gdp_calculated.where(d.gdp_calculated > 0))
LABEL.update({'log_mil': '군사비 (log10)', 'log_gdp': 'GDP (log10)'})

n_zero = int((d.TIV_5Y_Share == 0).sum())

# n은 모든 결과에 같이 적음
# 같은 상관계수여도 n이 작으면 유의하지 않고 n이 크면 유의할 수 있어서
print(f'N = {N}개국 | 수입점유율 0인 국가 = {n_zero}개 ({n_zero/N:.1%})')

N = 147개국 | 수입점유율 0인 국가 = 4개 (2.7%)


In [6]:
# 공통 함수: 상관계수 신뢰구간, 등가성 검정, 다중비교 보정

def corr_se(n, kind='spearman'):
    """Fisher z 변환한 상관계수의 표준오차 (근사식)

    Pearson:  z의 분산 ≈ 1/(n-3)
    Spearman: z의 분산 ≈ 1.06/(n-3)  (Fieller, Hartley & Pearson 1957)
    순위로 바꾸면서 정보가 조금 줄어들어서 Pearson보다 분산이 약간 큼
    """
    var = 1 / (n - 3) if kind == 'pearson' else 1.06 / (n - 3)
    return np.sqrt(var)


def corr_ci(r, n, alpha=ALPHA, kind='spearman'):
    """상관계수의 (1 - alpha) 신뢰구간. 02_EDA의 corr_ci랑 같은 계산

    r은 -1~1로 범위가 막혀 있어서 표본분포가 비대칭임
    z = arctanh(r)로 바꾸면 근사적으로 정규분포가 되니까
    z에서 구간을 잡고 tanh로 다시 r 단위로 되돌림
    """
    if n < 5 or abs(r) >= 1:
        return (np.nan, np.nan)
    z, se = np.arctanh(r), corr_se(n, kind)
    zc = stats.norm.ppf(1 - alpha / 2)
    lo, hi = np.tanh([z - zc * se, z + zc * se])
    return (float(lo), float(hi))


def tost_corr(r, n, bound=EQUIV_BOUND, alpha=ALPHA, kind='spearman'):
    """상관계수 등가성 검정 (TOST, Two One-Sided Tests)

    H0: |rho| >= bound   (무시할 수 없는 크기의 관계가 있음)
    H1: |rho| <  bound   (관계가 있어도 bound보다 작음 = 실무적으로 무관)

    통상검정이랑 가설 방향이 반대임
    통상검정은 "관계가 있다"를 보이는 거고, TOST는 "관계가 bound보다 작다"를 보이는 것

    단측검정 두 개를 둘 다 기각해야 H0를 기각함
      (1) H0: rho <= -bound  기각하면 rho > -bound
      (2) H0: rho >= +bound  기각하면 rho < +bound
    두 p-value 중에 큰 값이 TOST의 p-value

    반환: (p-value, (1 - 2*alpha) 신뢰구간)
      이 신뢰구간이 (-bound, +bound) 안에 완전히 들어가는 것과
      TOST가 유의수준 alpha에서 기각되는 것은 같은 조건임
      alpha = 0.05면 90% 구간
    """
    if n < 5 or abs(r) >= 1:
        return np.nan, (np.nan, np.nan)
    z, zb, se = np.arctanh(r), np.arctanh(bound), corr_se(n, kind)
    p_lower = stats.norm.sf((z + zb) / se)    # (1)
    p_upper = stats.norm.cdf((z - zb) / se)   # (2)
    zc = stats.norm.ppf(1 - alpha)
    lo, hi = np.tanh([z - zc * se, z + zc * se])
    return float(max(p_lower, p_upper)), (float(lo), float(hi))


def fdr(pvals):
    """Benjamini-Hochberg 보정. p-value 묶음을 넣으면 보정된 값(q)을 돌려줌

    검정을 여러 개 하면 실제로는 아무 관계가 없어도 그중 일부는 우연히 유의하게 나옴
    BH 보정은 유의하다고 판정한 것들 중에 잘못된 판정의 비율(FDR) 기댓값을 ALPHA 이하로 맞춰줌
    (검정들이 서로 독립이거나 양의 상관일 때 성립)
    q < ALPHA면 유의. q가 원래 p보다 작아지지는 않음
    """
    return multipletests(pvals, alpha=ALPHA, method='fdr_bh')[1]

---
## 2. 축 조합 검정

버블차트 두 개의 축 조합이 적절한지 확인

| 쌍 | x | y | 확인하려는 것 |
|---|---|---|---|
| 버블1 | 군사비/GDP | 분쟁위험도 | x축과 y축이 관계가 있는가 |
| 버블2 | 군사비/GDP | 수입점유율 | x축과 y축이 관계가 있는가 |
| 두 y축 | 분쟁위험도 | 수입점유율 | 두 차트가 같은 이야기를 반복하는가 |

쌍마다 검정을 두 개 한다

| 검정 | H0 | H1 | 기각하면 |
|---|---|---|---|
| 통상검정 | ρ = 0 | ρ ≠ 0 | 관계가 있다 |
| 등가성 검정 (TOST) | \|ρ\| ≥ 0.3 | \|ρ\| < 0.3 | 실무적으로 무관하다 |

**판정 규칙**

| 통상검정 | TOST | 판정 |
|---|---|---|
| 기각 | - | 관계 있음 |
| 기각 못 함 | 기각 | 무관 확정 |
| 기각 못 함 | 기각 못 함 | 판정 보류 |

In [7]:
# 축 조합 검정 (세 쌍 모두 같은 절차)
#
# 검정을 두 개 같이 돌리는 이유
#   통상검정에서 p가 크게 나왔다고 "관계가 없다"는 증거가 되는 건 아님
#   "관계가 있다는 증거를 못 찾았다"일 뿐이고, 표본이 부족해서 그런 걸 수도 있음
#   "무관하다"고 하려면 가설을 뒤집은 등가성 검정(TOST)을 따로 해야 함
#
# 세 쌍에 같은 절차를 쓰는 이유
#   쌍마다 검정 방법을 다르게 고르면 결과를 보고 방법을 정한 걸로 보일 수 있음
#   그래서 세 쌍 전부 같은 절차로 돌리고, 나온 판정에 따라 차트 형태를 정함
#
# 다중비교 보정
#   통상검정 3개, TOST 3개를 각각 한 묶음으로 보고 묶음 안에서 FDR 보정
#   판정은 보정된 q로 하고, 보정 전 p도 표에 같이 남김
#
# TOST의 한계 (결과 읽을 때 주의)
#   n = 147, 등가 한계 0.3이면 표본 상관이 대략 |rho| < 0.17 이어야 TOST가 기각됨 (보정 전 기준)
#   그래서 표본 상관이 0.17~0.3 사이면 한계보다 작아도 "무관 확정"이 아니라 "판정 보류"로 나옴
#   "판정 보류"는 관계가 있다는 뜻도, 없다는 뜻도 아님

PAIRS = [
    ('share_gdp',          'human_hazard_score', '버블1  군사비/GDP × 분쟁위험도'),
    ('share_gdp',          'TIV_5Y_Share',       '버블2  군사비/GDP × 수입점유율'),
    ('human_hazard_score', 'TIV_5Y_Share',       '두 y축 분쟁위험도 × 수입점유율'),
]

rows = []
for x, y, name in PAIRS:
    s = d[[x, y]].dropna()
    n = len(s)

    # 통상검정 (H0: rho = 0). Spearman 순위상관이랑 p-value
    r, p_zero = stats.spearmanr(s[x], s[y])
    ci95 = corr_ci(r, n)

    # 등가성 검정 (H0: |rho| >= EQUIV_BOUND). 90% 신뢰구간도 같이 받음
    p_tost, ci90 = tost_corr(r, n)

    rows.append({
        '쌍': name, 'n': n, 'rho': round(r, 3),
        '95% CI': '[{:.3f}, {:.3f}]'.format(*ci95),
        '90% CI': '[{:.3f}, {:.3f}]'.format(*ci90),
        '통상 p': p_zero, 'TOST p': p_tost,
    })

result = pd.DataFrame(rows)

# 다중비교 보정: 검정 종류별로 세 쌍을 한 묶음으로
result['통상 q'] = fdr(result['통상 p'])
result['TOST q'] = fdr(result['TOST p'])

# 판정 (미리 정한 규칙을 q에 적용)
result['판정'] = np.where(result['통상 q'] < ALPHA, '관계 있음',
                 np.where(result['TOST q'] < ALPHA, '무관 확정', '판정 보류'))

result.round({'통상 p': 4, 'TOST p': 4, '통상 q': 4, 'TOST q': 4})

,쌍,n,rho,95% CI,90% CI,통상 p,TOST p,통상 q,TOST q,판정
0,버블1 군사비/GDP × 분쟁위험도,147,0.158,"[-0.009, 0.316]","[0.018, 0.292]",0.0562,0.0399,0.0843,0.0598,판정 보류
1,버블2 군사비/GDP × 수입점유율,147,0.317,"[0.158, 0.459]","[0.185, 0.437]",0.0001,0.5851,0.0003,0.5851,관계 있음
2,두 y축 분쟁위험도 × 수입점유율,147,0.044,"[-0.124, 0.209]","[-0.097, 0.183]",0.5987,0.0010,0.5987,0.0029,무관 확정


In [8]:
# 판정에 따라 차트를 어떻게 그릴지 정리
# 같은 판정이어도 차트에서 의미하는 게 달라서 쌍별로 따로 해석함
# '판정 보류'는 관계가 없다는 게 아니라 결론을 못 낸다는 뜻이라 따로 처리

print('[검정 결과에 따른 차트 결정]\n')

for _, row in result.iterrows():
    print(f"[{row['쌍']}]")
    print(f"   rho={row['rho']}, n={row['n']}, 판정: {row['판정']}")

    if row['쌍'].startswith('버블1'):
        if row['판정'] == '관계 있음':
            print('   x축, y축이 같이 움직임. 산점도로 그리면 됨')
        elif row['판정'] == '무관 확정':
            print('   x축, y축은 실무적으로 무관함')
            print('   상관을 보여주는 그래프 말고 4분면 분류 맵으로 바꾸기')
            print('   (두 축이 무관하면 각 축 중앙값으로 나눈 네 분면에 국가가 고르게 들어감)')
        else:
            print('   관계가 있다고도 없다고도 말할 수 없음')
            print('   추세선 없이 4분면 분류 맵으로 그리고 캡션에 "관계 미확정"이라고 적기')

    elif row['쌍'].startswith('버블2'):
        if row['판정'] == '관계 있음':
            print('   x축, y축이 같이 움직임')
            print('   다만 국가 규모(GDP, 군사비) 때문인지는 3절 편상관에서 확인')
        else:
            print('   x축 재검토 필요 (군사비 로그 스케일 대안)')

    else:  # 두 y축
        if row['판정'] == '무관 확정':
            print('   두 y축이 서로 다른 정보를 담고 있음')
            print('   버블차트 두 개를 나란히 놓아도 됨')
        elif row['판정'] == '관계 있음':
            print('   두 차트가 같은 얘기를 함. 하나로 합치는 것 검토')
        else:
            print('   두 y축이 겹치는지 결론 못 냄. 캡션에 한계 적기')
    print()

[검정 결과에 따른 차트 결정]

[버블1  군사비/GDP × 분쟁위험도]
   rho=0.158, n=147, 판정: 판정 보류
   관계가 있다고도 없다고도 말할 수 없음
   추세선 없이 4분면 분류 맵으로 그리고 캡션에 "관계 미확정"이라고 적기

[버블2  군사비/GDP × 수입점유율]
   rho=0.317, n=147, 판정: 관계 있음
   x축, y축이 같이 움직임
   다만 국가 규모(GDP, 군사비) 때문인지는 3절 편상관에서 확인

[두 y축 분쟁위험도 × 수입점유율]
   rho=0.044, n=147, 판정: 무관 확정
   두 y축이 서로 다른 정보를 담고 있음
   버블차트 두 개를 나란히 놓아도 됨



---
## 3. 버블 크기 검정

버블차트는 x, y, 크기 세 채널로 정보를 전달한다. 크기로 쓰는 지표가 x축과 같은 것을 재고 있으면
같은 정보를 두 번 그리는 셈이 된다. x축인 군사비/GDP는 정의상 군사비를 GDP로 나눈 값이라
크기 후보(GDP, 군사비)와 겹칠 가능성이 구조적으로 있다.

| 검정 | H0 | H1 | 기각하면 |
|---|---|---|---|
| 1) x축 × 크기 통상검정 | ρ = 0 | ρ ≠ 0 | x축과 크기가 겹친다 (중복) |
| 1) x축 × 크기 등가성 검정 | \|ρ\| ≥ 0.3 | \|ρ\| < 0.3 | x축과 크기는 독립된 채널이다 |
| 2) 크기 통제 후 편상관 | ρ(x, y · 크기) = 0 | ≠ 0 | 크기가 같은 나라끼리 비교해도 x-y 관계가 남는다 |

In [9]:
# 버블 크기 검정. 차트 2개 × 크기 후보 2개 = 4개 조합
#
# 1) x축 × 크기: 2절이랑 같은 절차(통상검정 + TOST)로 분류
#      통상검정 기각              = 중복 (x축과 크기가 같이 움직임)
#      통상 기각 못함 + TOST 기각 = 독립 (실무적으로 무관)
#      둘 다 기각 못함            = 판정 보류
#    x축과 크기의 관계는 차트(y축)랑 상관없이 같아서 크기 후보 2개에 대해서만 검정
#
# 2) 편상관: 크기 지표의 영향을 빼고도 x-y 관계가 남는지 확인
#    단순 상관이 유의해도 그게 "큰 나라는 x도 크고 y도 크다"에서 온 거라면
#    크기를 통제했을 때 사라짐
#
# 다중비교 보정: 1)은 크기 후보 2개를, 2)는 4개 조합을 한 묶음으로 FDR 보정

def partial_corr(df, x, y, covar):
    """Spearman 편상관. 순위로 바꾼 다음 통제변수로 회귀하고 남은 잔차끼리의 Pearson 상관

    순위로 바꾸는 이유
      단순 상관을 Spearman으로 쟀으니까 편상관도 같은 척도여야 두 값을 비교할 수 있음
      그리고 군사비/GDP랑 수입점유율은 원본에 극단값이 있어서, 원본으로 계산하면 몇 나라가 결과를 좌우함

    p-value는 자유도 n - 2 - k 인 t 분포로 구함 (k = 통제변수 개수, 여기서는 1)
    순위에 적용한 거라 근사값
    """
    s = df[[x, y, covar]].dropna().rank()
    rx = smf.ols(f'{x} ~ {covar}', data=s).fit().resid   # x에서 통제변수로 설명되는 부분을 뺀 나머지
    ry = smf.ols(f'{y} ~ {covar}', data=s).fit().resid   # y도 마찬가지
    r, _ = stats.pearsonr(rx, ry)
    dof = len(s) - 3
    t = r * np.sqrt(dof / (1 - r ** 2))
    return r, t, 2 * stats.t.sf(abs(t), dof), dof


X_AXIS = 'share_gdp'
CHARTS = [('human_hazard_score', '버블1'), ('TIV_5Y_Share', '버블2')]
SIZES  = [('log_gdp', 'GDP'), ('log_mil', '군사비')]

# 1) x축 × 크기 (크기 후보별로 한 번씩)
rows = []
for size, size_name in SIZES:
    s = d[[X_AXIS, size]].dropna()
    r_xz, p_xz = stats.spearmanr(s[X_AXIS], s[size])
    p_tost_xz, _ = tost_corr(r_xz, len(s))
    rows.append({'버블 크기': size_name, 'x×크기 rho': round(r_xz, 3),
                 'x×크기 통상 p': p_xz, 'x×크기 TOST p': p_tost_xz})

channel = pd.DataFrame(rows)
channel['x×크기 통상 q'] = fdr(channel['x×크기 통상 p'])
channel['x×크기 TOST q'] = fdr(channel['x×크기 TOST p'])
channel['중복'] = np.where(channel['x×크기 통상 q'] < ALPHA, '중복',
                  np.where(channel['x×크기 TOST q'] < ALPHA, '독립', '판정 보류'))

# 2) 크기를 통제한 편상관 (4개 조합)
rows = []
for y, chart in CHARTS:
    for size, size_name in SIZES:
        s = d[[X_AXIS, y, size]].dropna()
        r_xy, _ = stats.spearmanr(s[X_AXIS], s[y])               # 통제 전 단순 상관
        r_pc, t_pc, p_pc, dof = partial_corr(s, X_AXIS, y, size)  # 통제 후 편상관
        rows.append({'차트': chart, '버블 크기': size_name, 'n': len(s),
                     '단순 rho': round(r_xy, 3), '편상관 rho': round(r_pc, 3), '편상관 p': p_pc})

size_test = pd.DataFrame(rows)
size_test['편상관 q'] = fdr(size_test['편상관 p'])
size_test['통제 후'] = np.where(size_test['편상관 q'] < ALPHA, '관계 유지', '유의하지 않음')

# 두 결과 합치기
size_test = size_test.merge(channel, on='버블 크기')
size_test = size_test[['차트', '버블 크기', 'n', 'x×크기 rho', 'x×크기 통상 q', 'x×크기 TOST q', '중복',
                       '단순 rho', '편상관 rho', '편상관 q', '통제 후']]
size_test.round(4)

,차트,버블 크기,n,x×크기 rho,x×크기 통상 q,x×크기 TOST q,중복,단순 rho,편상관 rho,편상관 q,통제 후
0,버블1,GDP,147,0.035,0.6766,0.0014,독립,0.158,0.161,0.0704,유의하지 않음
1,버블1,군사비,147,0.359,0.0000,0.7817,중복,0.158,0.172,0.0704,유의하지 않음
2,버블2,GDP,147,0.035,0.6766,0.0014,독립,0.317,0.473,0.0000,관계 유지
3,버블2,군사비,147,0.359,0.0000,0.7817,중복,0.317,0.020,0.8092,유의하지 않음


In [10]:
# 조합별 판정 출력
print('[버블 크기 선택]\n')

for _, r in size_test.iterrows():
    print(f"[{r['차트']}] 크기={r['버블 크기']}")
    print(f"   x축과 크기: rho={r['x×크기 rho']}, 통상 q={r['x×크기 통상 q']:.4f}, TOST q={r['x×크기 TOST q']:.4f}, 판정: {r['중복']}")

    if r['중복'] == '중복':
        print('   크기가 x축이랑 같이 움직임. 같은 정보를 두 번 그리게 됨')
    elif r['중복'] == '독립':
        print('   x축과 실무적으로 무관해서 써도 됨')
    else:
        print('   x축과 겹치는지 결론 못 냄')

    print(f"   편상관: 단순 {r['단순 rho']} / 통제 후 {r['편상관 rho']}, q={r['편상관 q']:.2e}, 판정: {r['통제 후']}")
    if r['통제 후'] == '관계 유지':
        print('   크기가 같은 나라끼리 비교해도 x-y 관계가 남음')
    else:
        print('   크기가 같은 나라끼리 비교하면 x-y 관계가 있다는 증거가 없음')
    print()

print('참고) 군사비/GDP = 군사비 ÷ GDP 라서 GDP를 통제한 편상관은')
print('      사실상 GDP가 같은 나라들끼리의 군사비 차이와 y의 관계를 보는 것')

[버블 크기 선택]

[버블1] 크기=GDP
   x축과 크기: rho=0.035, 통상 q=0.6766, TOST q=0.0014, 판정: 독립
   x축과 실무적으로 무관해서 써도 됨
   편상관: 단순 0.158 / 통제 후 0.161, q=7.04e-02, 판정: 유의하지 않음
   크기가 같은 나라끼리 비교하면 x-y 관계가 있다는 증거가 없음

[버블1] 크기=군사비
   x축과 크기: rho=0.359, 통상 q=0.0000, TOST q=0.7817, 판정: 중복
   크기가 x축이랑 같이 움직임. 같은 정보를 두 번 그리게 됨
   편상관: 단순 0.158 / 통제 후 0.172, q=7.04e-02, 판정: 유의하지 않음
   크기가 같은 나라끼리 비교하면 x-y 관계가 있다는 증거가 없음

[버블2] 크기=GDP
   x축과 크기: rho=0.035, 통상 q=0.6766, TOST q=0.0014, 판정: 독립
   x축과 실무적으로 무관해서 써도 됨
   편상관: 단순 0.317 / 통제 후 0.473, q=6.36e-09, 판정: 관계 유지
   크기가 같은 나라끼리 비교해도 x-y 관계가 남음

[버블2] 크기=군사비
   x축과 크기: rho=0.359, 통상 q=0.0000, TOST q=0.7817, 판정: 중복
   크기가 x축이랑 같이 움직임. 같은 정보를 두 번 그리게 됨
   편상관: 단순 0.317 / 통제 후 0.02, q=8.09e-01, 판정: 유의하지 않음
   크기가 같은 나라끼리 비교하면 x-y 관계가 있다는 증거가 없음

참고) 군사비/GDP = 군사비 ÷ GDP 라서 GDP를 통제한 편상관은
      사실상 GDP가 같은 나라들끼리의 군사비 차이와 y의 관계를 보는 것


---
## 4. 추세 검정

나라별로 지표가 시간에 따라 일관되게 늘거나 줄었는지 검정 (Mann-Kendall 검정)

- **H0**: 추세가 없다 (값의 크기가 시간 순서와 무관하다)
- **H1**: 한 방향으로 일관되게 늘거나 줄어든다

| 항목 | 내용 |
|---|---|
| 검정 방법 | Mann-Kendall 검정 + 자기상관 보정 (Hamed & Rao 1998) |
| 변화의 크기 | Sen 기울기 (모든 두 시점 쌍의 기울기 중앙값) |
| 대상 | 전체 국가 (지표별로 관측이 5개 이상인 나라) |
| 다중비교 | 지표별로 대상 국가 전체를 한 묶음으로 FDR 보정, q로 판정 |

In [11]:
# 숫자 한 줄을 받아서 "시간에 따라 일관된 방향이 있는지"만 답하는 함수
# 어느 나라인지, 무슨 지표인지는 모름

def mann_kendall(x, t=None, alpha=ALPHA):
    """Mann-Kendall 추세 검정 (자기상관 보정 포함)

    x: 시간순 값
    t: 각 값의 연도. 중간에 빠진 해가 있으면 꼭 넘겨야 함 (생략하면 0, 1, 2, ...)
    """
    x = np.asarray(x, float)
    t = np.arange(len(x), dtype=float) if t is None else np.asarray(t, float)
    keep = ~np.isnan(x)
    x, t = x[keep], t[keep]
    n = len(x)
    if n < MIN_OBS:
        # 관측이 4개면 전부 오르기만 해도 p ≈ 0.09 라서 유의수준 0.05에서는 절대 기각이 안 됨
        # 기각이 가능해지는 최소 개수가 5개
        return dict(n=n, S=np.nan, z=np.nan, p=np.nan, sen=np.nan, 보정계수=np.nan, 추세='표본부족')

    i, j = np.triu_indices(n, k=1)   # 모든 두 시점 쌍 (i가 앞, j가 뒤)

    # 검정통계량 S: 모든 쌍에서 (뒤 값 - 앞 값)의 부호를 더한 값
    #   오른 쌍은 +1, 내린 쌍은 -1, 같으면 0
    #   차이의 크기는 안 쓰고 부호만 써서 극단값 하나가 결과를 좌우하지 못함
    #   H0가 참이면 오른 쌍이랑 내린 쌍이 비슷해서 S의 기댓값은 0
    S = float(np.sign(x[j] - x[i]).sum())

    # S의 분산: H0가 참일 때 S가 얼마나 흔들리는지
    #   뒤의 뺄셈 항이 동점 보정. 같은 값이 여러 번 나오면 그 쌍들의 부호가 0이 돼서
    #   S가 움직일 수 있는 폭이 줄어드니까 분산도 그만큼 작게 잡아야 함
    #   (분쟁위험도처럼 소수점 한 자리 점수는 동점이 잘 생김)
    _, cnt = np.unique(x, return_counts=True)
    var = (n*(n-1)*(2*n+5) - (cnt*(cnt-1)*(2*cnt+5)).sum()) / 18
    if var == 0:
        # 값이 전부 같으면 분산이 0. 변화가 아예 없으니까 추세없음
        return dict(n=n, S=S, z=0.0, p=1.0, sen=0.0, 보정계수=1.0, 추세='추세없음')

    # Sen 기울기: 변화의 크기. 검정이 아니라 "한 해에 얼마씩 변했는지"를 재는 값
    #   모든 쌍의 기울기 (값 차이 ÷ 연도 차이) 중 중앙값
    #   중앙값이라서 최소제곱 회귀 기울기보다 극단값 영향을 덜 받음
    sen = float(np.median((x[j] - x[i]) / (t[j] - t[i])))

    # 자기상관 보정 (Hamed & Rao 1998)
    #   위 분산 공식은 연도별 값이 서로 독립이라고 가정함
    #   실제로는 올해 값이 작년 값이랑 비슷한 경우가 많고(양의 자기상관),
    #   이러면 S의 실제 분산이 공식보다 커서 보정 없이는 유의한 추세가 너무 많이 잡힘
    #   절차: 1) Sen 기울기로 추세를 뺌  2) 남은 값을 순위로 바꿈
    #         3) 순위의 자기상관 중에 유의한 것만 골라서 분산에 곱할 보정계수를 계산
    #   빠진 해가 있으면 시차를 관측 순서로 세기 때문에 이 보정은 근사임
    rank = stats.rankdata(x - sen * t)
    dev = rank - rank.mean()
    acov = np.correlate(dev, dev, 'full')[n-1:] / n   # 시차 0, 1, ..., n-1 의 자기공분산
    factor = 1.0
    if acov[0] > 0:
        rho = acov[1:] / acov[0]   # 시차 1 ~ n-1 의 자기상관
        k = np.arange(1, n)
        sig = np.abs(rho) > stats.norm.ppf(1 - alpha/2) / np.sqrt(n)   # 유의한 자기상관만 사용
        factor = 1 + 2 / (n*(n-1)*(n-2)) * ((n-k)*(n-k-1)*(n-k-2) * rho * sig).sum()
    # 보정계수가 1보다 작으면(음의 자기상관) 분산이 줄어서 오히려 유의해지기 쉬워짐
    # 짧은 계열에서는 자기상관 추정이 불안정해서, 유의성이 부풀려지지 않게 1 미만은 1로 둠
    # (원 논문에는 없고 보수적으로 가려고 넣은 것)
    factor = max(factor, 1.0)

    # 표준화: S를 표준정규분포랑 비교할 수 있게 z로 바꿈 (정규 근사)
    #   ±1은 연속성 보정. S는 띄엄띄엄한 값만 갖는데 정규분포는 연속이라 그 간격을 메워주는 것
    #   이 보정을 하면 |z|가 조금 작아져서 p가 조금 커짐. n이 작을수록 차이가 큼
    sd = np.sqrt(var * factor)
    z = 0.0 if S == 0 else (S - 1) / sd if S > 0 else (S + 1) / sd
    p = 2 * stats.norm.sf(abs(z))    # 증가든 감소든 '추세 있음'이라 양측 검정

    # 판정: 한 계열만 볼 때의 판정. p로 유의성을, z의 부호로 방향을 정함
    #   p >= alpha 면 z의 부호랑 상관없이 '추세없음'
    #   여러 나라를 한꺼번에 볼 때는 trend_table에서 FDR 보정한 q로 다시 판정함
    trend = '증가' if (p < alpha and z > 0) else '감소' if (p < alpha and z < 0) else '추세없음'
    return dict(n=n, S=S, z=z, p=p, sen=sen, 보정계수=factor, 추세=trend)

In [12]:
def trend_table(metric, countries=None, log=False, scale=1):
    """mann_kendall을 나라별로 돌리고 다중비교 보정까지 함

    기간은 갖고 있는 전체 기간을 씀
      - 기간을 줄이면 비교할 쌍의 수 n(n-1)/2 가 줄어서 추세를 찾아내는 힘(검정력)이 떨어짐
      - 특정 기간만 고르면 "왜 그 기간인가" 하는 자의적 선택 문제가 생김

    log=True:  군사비처럼 나라별 자릿수가 다른 지표
               로그를 취하면 Sen 기울기가 "연간 로그 증가분"이 돼서
               exp(기울기) - 1 로 연평균 증가율(%)을 구할 수 있음. 나라끼리 비교 가능
               로그는 순서를 안 바꾸니까 S는 그대로
               다만 자기상관 보정은 추세를 뺀 값으로 계산해서 z랑 p는 조금 달라질 수 있음
    log=False: 점수나 비율 지표. Sen 기울기가 "한 해에 몇 단위씩 변했는지"로 바로 읽힘
    scale:     표시 단위를 바꾸는 배수 (0~1 비율을 %로 볼 때 100). 검정 결과에는 영향 없음
    """
    pv = df_demand.pivot_table(index='Country', columns='Year', values=metric)
    years = sorted(c for c in pv.columns
                   if isinstance(c, (int, np.integer)) and pv[c].notna().any())
    if countries is not None:
        pv = pv.loc[pv.index.intersection(countries)]
    pv = pv[years]
    print(f'[{LABEL.get(metric, metric)}] 기간 {min(years)}~{max(years)} ({len(years)}년)')

    rows = []
    for c, row in pv.iterrows():   # 한 줄이 한 나라
        v = row.dropna().astype(float) * scale
        if log:
            v = np.log(v[v > 0])   # 0 이하는 로그가 안 돼서 제외
        if len(v) < MIN_OBS:
            continue
        r = mann_kendall(v.to_numpy(), t=v.index.to_numpy(dtype=float))   # 빠진 해가 있어도 실제 연도 간격으로 계산됨
        rec = {'국가': c, '관측': r['n'], 'MK z': round(r['z'], 2), '_z': r['z'], '_p': r['p'],
               '보정계수': round(r['보정계수'], 2)}   # 1이면 자기상관 보정 없음, 클수록 보정이 컸던 것
        if log:
            rec['연평균 %'] = round((np.exp(r['sen']) - 1) * 100, 1)
        else:
            rec['연간 변화'] = round(r['sen'], 4)
        rows.append(rec)

    t = pd.DataFrame(rows)
    if not len(t):
        return t

    # 다중비교 보정: 수십 개 나라를 동시에 검정하면 추세가 전혀 없어도 일부는 우연히 p < 0.05 가 됨
    #   (예: 50개국이 전부 추세가 없어도 평균 2~3개국은 우연히 유의하게 나옴)
    #   그래서 보정된 q로 판정하고, q < ALPHA 인 나라만 방향(증가/감소)을 표시
    t['q (FDR)'] = fdr(t['_p']).round(4)
    t['추세'] = np.where(t['q (FDR)'] >= ALPHA, '추세없음', np.where(t['_z'] > 0, '증가', '감소'))
    return t.drop(columns=['_z', '_p'])

In [14]:
# 지표마다 따로 검정함. 같은 나라여도 군사비는 늘었는데 분쟁위험도는 그대로일 수 있음
# (컬럼명, 로그 여부, 표시 배수). 표시 이름은 LABEL에서 가져옴
#
# 군사비는 물가 상승이랑 경제 성장만으로도 금액이 늘어남
# 그래서 물가·환율 영향을 안 받는 군사비/GDP를 같이 검정함
# 군사비는 늘었는데 군사비/GDP는 추세가 없으면 경제 규모가 커진 만큼만 늘어난 것
share_scale = 100 if df_demand.share_gdp.max() <= 1 else 1   # 0~1 비율이면 %로
TARGETS = [
    ('current_usd',        True,  1),             # 연평균 %
    ('share_gdp',          False, share_scale),   # 연간 변화 %p
    ('human_hazard_score', False, 1),             # 연간 변화 (점)
    ('TIV_5Y_Share',       False, 1),             # 연간 변화 %p
]

# 검정 대상: 전체 국가 (pool = None)
# 지표별로 관측이 MIN_OBS개 이상인 나라는 전부 검정하고, FDR 보정도 그 전체를 한 묶음으로 함
# 일부 국가만 보려면 국가명 목록을 넣으면 됨
#   예) 기준 연도 군사비 상위 50개국: df_demand[df_demand.Year == YEAR].nlargest(50, 'current_usd').Country
pool = None

TREND = {}
for metric, use_log, scale in TARGETS:
    name = LABEL[metric]
    t = trend_table(metric, countries=pool, log=use_log, scale=scale)
    if not len(t):
        print('  검정 가능한 국가 없음\n')
        continue

    n_sig = (t['추세'] != '추세없음').sum()
    print(f'  유의한 추세 (FDR 보정 후): {n_sig} / {len(t)}개국')
    print('  ', t['추세'].value_counts().to_dict())

    sort_col = '연평균 %' if use_log else '연간 변화'
    display(t.sort_values(sort_col, ascending=False).head(10))
    TREND[name] = t.set_index('국가')
    print()

# 사용 예: TREND['군사비'].loc['Poland', '추세']

[군사비] 기간 2000~2025 (26년)
  유의한 추세 (FDR 보정 후): 132 / 162개국
   {'증가': 132, '추세없음': 30}


,국가,관측,MK z,보정계수,연평균 %,q (FDR),추세
62,Haiti,13,3.84,1.00,16.8,0.0003,증가
120,Qatar,11,3.58,1.00,16.4,0.0007,증가
132,Somalia,13,4.21,1.00,16.2,0.0001,증가
8,Azerbaijan,26,3.01,2.99,15.5,0.0039,증가
108,Niger,23,5.19,1.40,13.9,0.0000,증가
92,Mali,26,6.35,1.19,13.5,0.0000,증가
152,United Arab Emirates,15,3.67,1.47,13.1,0.0005,증가
21,Burkina Faso,26,5.05,1.83,13.1,0.0000,증가
23,Cambodia,26,4.48,2.17,12.7,0.0000,증가
5,Armenia,26,4.23,2.19,12.2,0.0001,증가



[군사비/GDP] 기간 2000~2025 (26년)
  유의한 추세 (FDR 보정 후): 60 / 165개국
   {'추세없음': 105, '감소': 36, '증가': 24}


,국가,관측,MK z,보정계수,연간 변화,q (FDR),추세
2,Algeria,26,3.55,1.63,0.1773,0.0022,증가
8,Azerbaijan,26,4.72,1.00,0.1214,0.0000,증가
94,Mali,26,2.34,2.95,0.1112,0.0518,추세없음
5,Armenia,26,4.98,1.00,0.1028,0.0000,증가
104,Myanmar,19,2.31,1.00,0.0966,0.0548,추세없음
154,Ukraine,26,2.91,1.70,0.0962,0.0134,증가
135,Somalia,13,3.72,1.00,0.0861,0.0013,증가
125,Russia,26,3.79,1.10,0.0686,0.0010,증가
21,Burkina Faso,26,2.46,3.28,0.0605,0.0389,증가
89,Lithuania,26,1.47,3.11,0.0598,0.2341,추세없음



[분쟁위험도] 기간 2017~2025 (9년)
  유의한 추세 (FDR 보정 후): 29 / 167개국
   {'추세없음': 138, '증가': 19, '감소': 10}


,국가,관측,MK z,보정계수,연간 변화,q (FDR),추세
43,Ecuador,9,3.14,1.0,0.8000,0.0198,증가
21,Burkina Faso,9,3.65,1.0,0.6775,0.0190,증가
64,Haiti,9,3.44,1.0,0.6000,0.0190,증가
14,Benin,9,3.35,1.0,0.5143,0.0190,증가
49,Eswatini,9,2.73,1.0,0.4071,0.0447,증가
150,Togo,9,3.23,1.0,0.4071,0.0198,증가
119,Papua New Guinea,9,2.64,1.0,0.3542,0.0484,증가
138,South Africa,9,3.14,1.0,0.3524,0.0198,증가
133,Sierra Leone,9,2.42,1.0,0.3000,0.0674,추세없음
96,Mali,9,3.23,1.0,0.3000,0.0198,증가



[수입점유율] 기간 2004~2025 (22년)
  유의한 추세 (FDR 보정 후): 37 / 176개국
   {'추세없음': 139, '증가': 20, '감소': 17}


,국가,관측,MK z,보정계수,연간 변화,q (FDR),추세
133,Saudi Arabia,22,2.63,2.19,0.4629,0.0455,증가
129,Qatar,22,3.56,2.68,0.2678,0.0103,증가
75,Iraq,22,0.69,3.27,0.1077,0.6726,추세없음
86,Kuwait,22,3.03,2.22,0.0849,0.0224,증가
80,Japan,22,1.03,2.36,0.0666,0.5052,추세없음
49,Egypt,22,0.81,2.14,0.0659,0.6124,추세없음
7,Australia,22,1.83,1.10,0.0651,0.1843,추세없음
126,Philippines,22,3.47,2.29,0.0558,0.0103,증가
73,Indonesia,22,1.33,2.87,0.0539,0.3574,추세없음
121,Pakistan,22,1.05,1.65,0.0486,0.4951,추세없음
